# Dual-batch label-suspicion audit (audios4 + audios5)

Generalises §11 of `honest_eval_and_improve.ipynb` from a5-only to **both a4 and a5**, with the §10 **PASS 1 (full a2 + SPW_DEPLOY) / PASS 2 (balanced a2)** treatment so we don't blame a flipped label on a 67% positive training prior.

## Why the dual pass matters
audios2 is 147+/72− (~67% positive). A model trained on it is biased toward predicting *cheating* — so a row labelled honest scoring high under PASS 1 may just be the prior, not a real disagreement. A row that disagrees under **both** PASS 1 (full a2, SPW_DEPLOY=4.88 calibrating to deployment 17%) and PASS 2 (balanced 72/72) is a robust suspect.

## Why both batches
§11 only audits a5. We need a4 too because:
1. If a4 has comparable suspect-rate to a5, the Rot B test↔CV gap is a *prior shift / training-distribution* issue, not a5 noise — relabelling a5 won't close the gap.
2. If a5 suspect-rate ≫ a4, §10's hypothesis-1 (a5 label noise) is supported and the relabel work is justified.
3. Numbers from each side give you a per-batch shortlist for re-listening.

## Models used
Three independent base models, each trained on a2 only — then aggregated:
- `whisper_wp_xgb` — whisper-medium frozen embeddings (acoustic, batch-robust)
- `wavlm_whole_pre` — WavLM pretrained whole-pool **(prefer over `_ft` per memory: pre > ft at 540 labels)**
- `text_all` — 55 text/disfluency/pause features (independent modality)

Suspect = (acoustic models agree) AND (both passes agree). Text is shown as a third signal but **acoustic agreement carries the verdict**, since text features can be confounded by topic/lexicon shift across batches.

## Output
- Per-batch summary (counts and rates of suspects, both directions)
- Top-K shortlists with `audio_path` for direct re-listen
- Per-candidate aggregate (which speakers to re-listen first)
- CSVs under `checkpoints_label_audit/`
- Cross-batch comparison cell that maps directly onto §10's three hypotheses

In [ ]:
# === 0. Imports and constants ===
from pathlib import Path
import re, warnings
import numpy as np
import pandas as pd

import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings('ignore')

NB_DIR   = Path('.').resolve()
SAVE_DIR = NB_DIR / 'checkpoints_label_audit'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BATCHES = ['audios2', 'audios4', 'audios5']

DEPLOY_POS_RATE = 0.17
SPW_DEPLOY      = (1.0 - DEPLOY_POS_RATE) / DEPLOY_POS_RATE  # 4.88

LABEL_MAP = {
    'read':1,'cheating':1,'reading':1,'scripted':1,'yes':1,'1':1,1:1,
    'spontaneous':0,'not cheating':0,'not_cheating':0,'no':0,'0':0,0:0,'genuine':0,
}

ALL_TEXT_FEATURES = [
    'filler_rate','filler_count','repetition_rate','repair_rate','discourse_marker_rate','hedge_rate',
    'ttr','mattr','mtld','complex_word_rate','avg_word_length','n_words','n_unique_words',
    'avg_sentence_length','std_sentence_length','fragment_rate','n_sentences','self_ref_rate',
    'noun_rate','verb_rate','adj_rate',
    'pause_mean','pause_std','pause_median','pause_skew','long_pause_rate','pause_ratio','n_pauses',
    'pause_regularity','pause_before_content_ratio','pause_before_function_ratio','mid_phrase_pause_rate',
    'words_per_sec','articulation_rate','initial_pause','longest_pause',
    'suspicious_gap_count','suspicious_gap_ratio',
    'formal_transition_count','formal_transition_rate','ai_phrase_count','ai_phrase_rate',
    'f0_mean','f0_std','f0_range','f0_skew','f0_slope','energy_mean','energy_std','speaking_rate_std',
    'jitter_local','shimmer_local','hnr_mean',
    'mean_perplexity','burstiness',
]

print(f'NB_DIR   : {NB_DIR}')
print(f'SAVE_DIR : {SAVE_DIR}')
print(f'SPW_DEPLOY = {SPW_DEPLOY:.2f}  (deploy positive rate {DEPLOY_POS_RATE:.0%})')

In [ ]:
# === 1. Data loading (mirrors honest_eval §2) ===
WAVLM_CSV_CANDIDATES = {
    'wpre': lambda n: [f'{n}_wavlm_whole.csv',     f'{n}_whole_pretrained.csv'],
    'wft' : lambda n: [f'{n}_whole_finetuned.csv'],
    'spre': lambda n: [f'{n}_wavlm_seg.csv',       f'{n}_seg_pretrained.csv'],
    'sft' : lambda n: [f'{n}_seg_finetuned.csv'],
}

def _first_existing(cands):
    for c in cands:
        p = NB_DIR / c
        if p.exists(): return p
    return None

def load_gt(name):
    gt = pd.read_csv(NB_DIR / f'{name}GT.csv')
    fn_col  = next(c for c in gt.columns if c.lower() in ('filename','file','name'))
    lbl_col = next(c for c in gt.columns if c.lower() in ('label','class','cheating','gt','label_int','ground_truth'))
    gt = gt.rename(columns={fn_col:'filename', lbl_col:'label_raw'})
    gt['label_int'] = gt['label_raw'].map(
        lambda x: LABEL_MAP.get(x, LABEL_MAP.get(str(x).lower().strip(), -1)))
    return gt[gt['label_int'].isin([0,1])][['filename','label_int']]

def load_folder(name):
    gt   = load_gt(name)
    text = pd.read_csv(NB_DIR / f'{name}_features.csv')
    df   = gt.merge(text, on='filename', how='inner')
    for tag, cand_fn in WAVLM_CSV_CANDIDATES.items():
        path = _first_existing(cand_fn(name))
        if path is not None:
            sub = pd.read_csv(path)
            sub = sub.rename(columns={c: (c if c == 'filename' else f'{c}_{tag}') for c in sub.columns})
            df = df.merge(sub, on='filename', how='inner')
    whr = pd.read_csv(NB_DIR / f'{name}_whisper_whole.csv')
    df = df.merge(whr.rename(columns={c: (c if c == 'filename' else f'{c}_whisper') for c in whr.columns}),
                  on='filename', how='inner')
    df['batch'] = name
    return df

batches = {b: load_folder(b) for b in BATCHES}

# Attach candidate_id (filename pattern: <cid>_<question>.<ext>)
_RE_CAND = re.compile(r'^(.+)_(\d{1,3})\.[a-zA-Z0-9]+$')
for b, df in batches.items():
    df['candidate_id'] = df['filename'].astype(str).map(
        lambda f: (_RE_CAND.match(f).group(1) if _RE_CAND.match(f) else None))
    df['question'] = df['filename'].astype(str).map(
        lambda f: (_RE_CAND.match(f).group(2) if _RE_CAND.match(f) else None))

for b, df in batches.items():
    y = df['label_int'].values
    print(f'  {b}: n={len(df):4d}  cheat={int((y==1).sum()):3d}  honest={int((y==0).sum()):3d}  '
          f'pos_rate={(y==1).mean():.2f}')

In [ ]:
# === 2. Build BASE_REGISTRY (mirrors honest_eval §2) ===
first = batches[BATCHES[0]]
WPRE_COLS = [c for c in first.columns if (c.startswith('wavlm_') or c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')) and c.endswith('_wpre')]
WFT_COLS  = [c for c in first.columns if (c.startswith('wavlm_') or c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')) and c.endswith('_wft')]
SPRE_COLS = [c for c in first.columns if (c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')) and c.endswith('_spre')]
SFT_COLS  = [c for c in first.columns if (c.startswith('wavlm_mean_') or c.startswith('wavlm_std_')) and c.endswith('_sft')]
WH_COLS   = [c for c in first.columns if c.startswith('whisper_') and c.endswith('_whisper')]
TEXT_ALL  = [c for c in ALL_TEXT_FEATURES if c in first.columns]

def make_xgb(n_feats, seed=42):
    colsample = 0.3 if n_feats > 500 else 0.8
    return xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=colsample, min_child_weight=3,
        scale_pos_weight=float(SPW_DEPLOY), eval_metric='logloss',
        random_state=seed)

def mk_X(cols): return lambda d: d[cols].fillna(0).values

BASE_REGISTRY = {
    'whisper_wp_xgb': (mk_X(WH_COLS), lambda s=42: make_xgb(len(WH_COLS), s)),
}
if WPRE_COLS: BASE_REGISTRY['wavlm_whole_pre'] = (mk_X(WPRE_COLS), lambda s=42: make_xgb(len(WPRE_COLS), s))
if WFT_COLS:  BASE_REGISTRY['wavlm_whole_ft']  = (mk_X(WFT_COLS),  lambda s=42: make_xgb(len(WFT_COLS), s))
if SPRE_COLS: BASE_REGISTRY['wavlm_seg_pre']   = (mk_X(SPRE_COLS), lambda s=42: make_xgb(len(SPRE_COLS), s))
if SFT_COLS:  BASE_REGISTRY['wavlm_seg_ft']    = (mk_X(SFT_COLS),  lambda s=42: make_xgb(len(SFT_COLS), s))
BASE_REGISTRY['text_all']   = (mk_X(TEXT_ALL),   lambda s=42: make_xgb(len(TEXT_ALL), s))

print('Available models:', list(BASE_REGISTRY))
print(f'  whisper d={len(WH_COLS)}   wavlm_pre d={len(WPRE_COLS)}   wavlm_ft d={len(WFT_COLS)}   text d={len(TEXT_ALL)}')

In [ ]:
# === 3. Auto-pick the 3 strongest base models (a2-trained, eval on a4+a5 combined AUC) ===
# The 'best' models are the ones whose a2-only training transfers best off-batch.
# We rank by mean AUC across (a4, a5) under the BALANCED-pass (prior-clean) training.

def _train_a2(name, balanced, seed=42):
    X_fn, factory = BASE_REGISTRY[name]
    df_tr = batches['audios2']
    if balanced:
        n_neg = int((df_tr['label_int']==0).sum())
        pos = df_tr[df_tr['label_int']==1].sample(n=n_neg, random_state=seed)
        neg = df_tr[df_tr['label_int']==0]
        df_tr = pd.concat([pos, neg], ignore_index=True)
    Xtr = X_fn(df_tr); ytr = df_tr['label_int'].values
    sc = StandardScaler().fit(Xtr)
    clf = factory(seed)
    clf.fit(sc.transform(Xtr), ytr)
    return clf, sc, X_fn

rank_rows = []
for name in BASE_REGISTRY:
    try:
        clf, sc, X_fn = _train_a2(name, balanced=True, seed=42)
        aucs = {}
        for tgt in ['audios4','audios5']:
            df_e = batches[tgt]
            p = clf.predict_proba(sc.transform(X_fn(df_e)))[:,1]
            aucs[tgt] = roc_auc_score(df_e['label_int'].values, p)
        rank_rows.append({'model': name, 'AUC_a4': round(aucs['audios4'],3),
                          'AUC_a5': round(aucs['audios5'],3),
                          'mean_AUC': round(np.mean(list(aucs.values())),3)})
    except Exception as e:
        print(f'  skip {name}: {e}')

rank_df = pd.DataFrame(rank_rows).sort_values('mean_AUC', ascending=False).reset_index(drop=True)
print('=== Base-model transfer ranking (balanced-a2 train, eval on each target batch) ===')
print(rank_df.to_string(index=False))

# Pick top-3 — but enforce that whisper + wavlm_pre are in there if available, since
# they're the two independent acoustic signals the verdict relies on.
MUST_INCLUDE = [m for m in ('whisper_wp_xgb','wavlm_whole_pre') if m in BASE_REGISTRY]
ranked = rank_df['model'].tolist()
AUDIT_MODELS = list(MUST_INCLUDE)
for m in ranked:
    if m not in AUDIT_MODELS and len(AUDIT_MODELS) < 3:
        AUDIT_MODELS.append(m)
print(f'\nAudit models (locked in for next cells): {AUDIT_MODELS}')

## Audit method (the rule the next cells implement)

For each row in the target batch (a4 or a5):

1. Score it under each `(model, pass)` pair where pass ∈ {full a2 + SPW_DEPLOY, balanced a2}.  
   With 3 models × 2 passes = **6 scores per row**.
2. **Acoustic verdict** = mean of the two acoustic models' (whisper, wavlm_pre) two-pass averages → 2 numbers.
3. **Suspect rules** (per row):
    - `pos_label_likely_honest` (label says cheating, models say honest):  
      `label_int==1` AND **both** acoustic two-pass means < `POS_THR` (default 0.30)
    - `neg_label_likely_cheating` (label says honest, models say cheating):  
      `label_int==0` AND **both** acoustic two-pass means > `NEG_THR` (default 0.70)
4. **Strength tier**:
    - `strong` — text_all also agrees with acoustic (all 3 modalities point the same way)
    - `acoustic_only` — acoustic agrees but text disagrees → the row may be a topic/lexicon shift artifact for text; the acoustic verdict still stands but is graded one notch lower in confidence
5. **Disagreement score** = `|label_int - mean(all 6 scores)|`, used to rank within each suspect bucket.

The thresholds 0.30 / 0.70 mirror §11. They are intentionally strict — a borderline row (e.g. 0.45) is *not* a label issue, it's a hard case for the model.

In [ ]:
# === 4. Score+audit functions ===
POS_THR = 0.30   # both acoustic means < this on labeled-cheating row -> suspect honest
NEG_THR = 0.70   # both acoustic means > this on labeled-honest   row -> suspect cheating
SHOW    = 15     # rows printed per shortlist

ACOUSTIC = [m for m in ('whisper_wp_xgb','wavlm_whole_pre') if m in AUDIT_MODELS]
TEXT_M   = [m for m in ('text_all',) if m in AUDIT_MODELS]
assert len(ACOUSTIC) == 2, f'Need both whisper + wavlm_whole_pre for the verdict. Got {ACOUSTIC}'

def _score_one(name, target, balanced, seed=42):
    clf, sc, X_fn = _train_a2(name, balanced=balanced, seed=seed)
    df = batches[target]
    p = clf.predict_proba(sc.transform(X_fn(df)))[:,1]
    return p

def score_target(target):
    """Return per-row score table with one column per (model, pass)."""
    df = batches[target][['filename','candidate_id','question','label_int']].copy()
    for name in AUDIT_MODELS:
        df[f'{name}__full'] = _score_one(name, target, balanced=False)
        df[f'{name}__bal']  = _score_one(name, target, balanced=True)
    # Per-model two-pass means (the row's per-model verdict)
    for name in AUDIT_MODELS:
        df[f'{name}__mean'] = (df[f'{name}__full'] + df[f'{name}__bal']) / 2
    # Acoustic verdict and grand mean
    df['acoustic_min'] = df[[f'{m}__mean' for m in ACOUSTIC]].min(axis=1)
    df['acoustic_max'] = df[[f'{m}__mean' for m in ACOUSTIC]].max(axis=1)
    df['acoustic_avg'] = df[[f'{m}__mean' for m in ACOUSTIC]].mean(axis=1)
    if TEXT_M:
        df['text_avg'] = df[[f'{m}__mean' for m in TEXT_M]].mean(axis=1)
        df['all_avg']  = df[[f'{m}__mean' for m in AUDIT_MODELS]].mean(axis=1)
    else:
        df['text_avg'] = np.nan
        df['all_avg']  = df['acoustic_avg']
    df['disagree'] = (df['label_int'] - df['all_avg']).abs()
    df['audio_path'] = [str(NB_DIR / target / str(c) / str(f))
                        for c, f in zip(df['candidate_id'], df['filename'])]
    return df

def _suspect_table(scored, target):
    s = scored.copy()
    pos_rule = (s['label_int']==1) & (s['acoustic_max'] < POS_THR)
    neg_rule = (s['label_int']==0) & (s['acoustic_min'] > NEG_THR)
    s['suspect_type'] = ''
    s.loc[pos_rule, 'suspect_type'] = 'pos_label_likely_honest'
    s.loc[neg_rule, 'suspect_type'] = 'neg_label_likely_cheating'
    if 'text_avg' in s and s['text_avg'].notna().any():
        text_agrees_pos = pos_rule & (s['text_avg'] < POS_THR)
        text_agrees_neg = neg_rule & (s['text_avg'] > NEG_THR)
        s['strength'] = ''
        s.loc[pos_rule, 'strength'] = 'acoustic_only'
        s.loc[text_agrees_pos, 'strength'] = 'strong'
        s.loc[neg_rule, 'strength'] = 'acoustic_only'
        s.loc[text_agrees_neg, 'strength'] = 'strong'
    else:
        s['strength'] = np.where(s['suspect_type']=='', '', 'acoustic_only')
    return s

def _print_top(df, sort_col, asc, title):
    if not len(df):
        print(f'\n  (no rows for: {title})'); return
    cols = ['candidate_id','question','filename','label_int',
            'whisper_wp_xgb__mean','wavlm_whole_pre__mean','text_avg',
            'acoustic_avg','strength','audio_path']
    cols = [c for c in cols if c in df.columns]
    out = df.sort_values(sort_col, ascending=asc).head(SHOW)[cols].copy()
    for c in cols:
        if out[c].dtype == float: out[c] = out[c].round(3)
    print(f'\n--- TOP {SHOW}: {title} ---')
    with pd.option_context('display.max_columns', None, 'display.width', 260, 'display.max_colwidth', 80):
        print(out.to_string(index=False))

def audit(target):
    print('='*82); print(f' LABEL AUDIT — target = {target}'); print('='*82)
    scored = score_target(target)
    s = _suspect_table(scored, target)

    n_pos = int((s['label_int']==1).sum())
    n_neg = int((s['label_int']==0).sum())
    n_pos_susp = int((s['suspect_type']=='pos_label_likely_honest').sum())
    n_neg_susp = int((s['suspect_type']=='neg_label_likely_cheating').sum())
    n_pos_strong = int(((s['suspect_type']=='pos_label_likely_honest') & (s['strength']=='strong')).sum())
    n_neg_strong = int(((s['suspect_type']=='neg_label_likely_cheating') & (s['strength']=='strong')).sum())

    pos_rate = n_pos_susp / max(n_pos,1) * 100
    neg_rate = n_neg_susp / max(n_neg,1) * 100
    print(f'  total rows scored      : {len(s)}')
    print(f'  labeled cheating       : {n_pos:4d}   suspect honest   : {n_pos_susp:3d}  ({pos_rate:5.1f}%)   strong: {n_pos_strong}')
    print(f'  labeled honest         : {n_neg:4d}   suspect cheating : {n_neg_susp:3d}  ({neg_rate:5.1f}%)   strong: {n_neg_strong}')
    print(f'  thresholds  POS_THR={POS_THR}  NEG_THR={NEG_THR}  (acoustic two-pass means)')

    pos_df = s[s['label_int']==1]
    neg_df = s[s['label_int']==0]
    _print_top(pos_df, 'acoustic_max', True,
               f'{target} LABELED CHEATING but both acoustic models score lowest (label may be honest)')
    _print_top(neg_df, 'acoustic_min', False,
               f'{target} LABELED HONEST but both acoustic models score highest (label may be cheating)')

    cand = s.assign(disagree=(s['label_int']-s['all_avg']).abs()).groupby('candidate_id').agg(
        n=('filename','size'),
        labels=('label_int', lambda x: ','.join(map(str, sorted(x.unique())))),
        n_suspect=('suspect_type', lambda x: int((x!='').sum())),
        mean_disagree=('disagree','mean'),
    ).sort_values(['n_suspect','mean_disagree'], ascending=[False, False]).head(8)
    print(f'\n--- TOP 8 candidates by suspect-audio count + disagreement ({target}) ---')
    print(cand.round(3).to_string())

    cols_save = ['candidate_id','question','filename','label_int',
                 'whisper_wp_xgb__full','whisper_wp_xgb__bal','whisper_wp_xgb__mean',
                 'wavlm_whole_pre__full','wavlm_whole_pre__bal','wavlm_whole_pre__mean',
                 'text_avg','acoustic_min','acoustic_max','acoustic_avg','all_avg',
                 'suspect_type','strength','audio_path']
    cols_save = [c for c in cols_save if c in s.columns]
    p_pos = SAVE_DIR / f'{target}_audit_positives.csv'
    p_neg = SAVE_DIR / f'{target}_audit_negatives.csv'
    p_all = SAVE_DIR / f'{target}_audit_all.csv'
    pos_df.sort_values('acoustic_max').to_csv(p_pos, columns=cols_save, index=False)
    neg_df.sort_values('acoustic_min', ascending=False).to_csv(p_neg, columns=cols_save, index=False)
    s.assign(disagree=(s['label_int']-s['all_avg']).abs()).sort_values('disagree', ascending=False) \
      .to_csv(p_all, columns=cols_save+['disagree'] if 'disagree' not in cols_save else cols_save, index=False)
    print(f'\nSaved:')
    print(f'  {p_pos}  ({len(pos_df)} rows, sort by acoustic_max ASC)')
    print(f'  {p_neg}  ({len(neg_df)} rows, sort by acoustic_min DESC)')
    print(f'  {p_all}  ({len(s)} rows, sort by joint disagreement DESC)')
    return {'target': target, 'n': len(s), 'n_pos': n_pos, 'n_neg': n_neg,
            'n_pos_susp': n_pos_susp, 'n_neg_susp': n_neg_susp,
            'n_pos_strong': n_pos_strong, 'n_neg_strong': n_neg_strong,
            'pos_rate_pct': round(pos_rate,1), 'neg_rate_pct': round(neg_rate,1),
            'scored': s}

In [ ]:
# === 5. Run audit on audios4 ===
audit_a4 = audit('audios4')

In [ ]:
# === 6. Run audit on audios5 ===
audit_a5 = audit('audios5')

In [ ]:
# === 7. Cross-batch comparison — maps onto §10's three hypotheses ===
summary = pd.DataFrame([
    {k: v for k, v in audit_a4.items() if k != 'scored'},
    {k: v for k, v in audit_a5.items() if k != 'scored'},
])
print('='*82); print(' CROSS-BATCH SUMMARY'); print('='*82)
print(summary.to_string(index=False))
summary.to_csv(SAVE_DIR / 'cross_batch_summary.csv', index=False)

a4_pos_rate = audit_a4['pos_rate_pct']
a5_pos_rate = audit_a5['pos_rate_pct']
a4_neg_rate = audit_a4['neg_rate_pct']
a5_neg_rate = audit_a5['neg_rate_pct']
delta_pos = a5_pos_rate - a4_pos_rate
delta_neg = a5_neg_rate - a4_neg_rate

print('\n=== Verdict mapping onto §10 hypotheses ===')
print(f'  a4: {a4_pos_rate:.1f}% positives suspect-honest, {a4_neg_rate:.1f}% negatives suspect-cheating')
print(f'  a5: {a5_pos_rate:.1f}% positives suspect-honest, {a5_neg_rate:.1f}% negatives suspect-cheating')
print(f'  delta(a5 - a4) on positive-side suspect rate: {delta_pos:+.1f}pp')
print(f'  delta(a5 - a4) on negative-side suspect rate: {delta_neg:+.1f}pp')

if delta_pos >= 5 or delta_neg >= 5:
    print('\n  -> a5 has materially more suspect labels than a4.')
    print('     Supports §10 HYPOTHESIS 1 (a5 label noise drives the Rot B gap).')
    print('     Action: relabel a5 from the strong+acoustic_only suspects, then re-run')
    print('     §6c rotation B in honest_eval. Expected: gap_f1 shrinks toward 0.')
elif abs(delta_pos) < 3 and abs(delta_neg) < 3:
    print('\n  -> a4 and a5 have comparable suspect rates.')
    print('     Label noise is symmetric; the Rot B gap is NOT a5-specific noise.')
    print('     More likely: training-prior shift (audios2 67% pos -> a4 15% pos -> a5 17% pos),')
    print('     or a4 cheaters are intrinsically subtler (HYPOTHESIS 2). Relabelling will')
    print('     not close the gap by itself — also try per-batch standardisation, group-CV,')
    print('     or rebalanced training.')
else:
    print('\n  -> mixed signal. Inspect the per-direction rates above and the saved CSVs')
    print('     before deciding whether to relabel.')

## 8. Ground-truth audit playbook (read this before re-listening)

### What counts as label noise here
Three things look the same in the data but need different treatment:

| Class | Definition | Action on re-listen |
|---|---|---|
| **Hard mislabel** | Audio is unambiguously the *opposite* class on listen (clearly fluent extemporaneous speech labelled `cheating`, or clearly read-from-script labelled `honest`). | Flip the label. |
| **Onset-shift** | Audio starts spontaneous and turns into reading mid-clip (or vice versa). Per project memory, *cheating can start mid-exam* — the per-audio label is correct if **any segment is reading**. | Keep label. Note `onset_seconds` in a side column for future segment-level work. |
| **Genuine ambiguity** | Slow, careful, prepared speech that's neither obviously read nor obviously extempore. | Add to a separate `unclear` set and **exclude from validation/test**. Do not relabel by guess. Keeping noise out of evaluation matters more than keeping it out of training. |

### What is *not* label noise
- **Hard cases for the model** — the audio is correctly labelled but acoustically subtle (e.g. someone who reads with natural prosody). The model just can't find it. This shows up as scores in the 0.40–0.60 band and is filtered out by `POS_THR=0.30 / NEG_THR=0.70`. Don't relabel these.
- **Class-policy edge cases** — the candidate read carefully prepared notes but spoke them in their own words. Whether this is "cheating" depends on *task instructions to the candidate*, not on the audio. Confirm the task-rules definition before flipping anything.

### Re-listen protocol (~30 sec/audio)
1. Open `<batch>_audit_positives.csv`, sort by `acoustic_max` ascending — strongest-suspect-honest rows on top.
2. Listen to **first 15 sec + last 15 sec** of each (cheating onset can be anywhere; both ends covers it cheaply).
3. For each row, write one of `flip_to_0`, `flip_to_1`, `keep`, `unclear` into a `decision` column.
4. Stop after the first 5 consecutive `keep` decisions in a sorted CSV — past that point, suspects are weaker than the model's noise floor and your time is better spent on the negative-side CSV.
5. Repeat for `<batch>_audit_negatives.csv`.

### Adjudication when only acoustic agrees (text disagrees)
Rows tagged `acoustic_only` (not `strong`) are candidates where:
- both whisper and wavlm agree the label looks wrong, BUT
- the text features point the other way.

Most likely cause: text features are picking up topic/lexicon shift across batches (a4 vs a5 prompts differ in vocabulary distribution), not cheating signal. **Default trust the acoustic verdict** but listen with extra care — if the audio sounds borderline, mark `unclear` rather than flipping.

### Closing the test↔CV gap
After you have decisions for the suspect set:

1. Apply the flips to the GT CSVs (write `audios5GT_v2.csv`, etc., never overwrite the originals).
2. Move `unclear` rows into a separate `unclear_excluded.csv` and drop them from the evaluation set (keep them in train if you want, with `sample_weight=0` for evaluation).
3. Re-run §6c (rotation B) in `honest_eval_and_improve.ipynb` against the new GT files.
4. The expected outcomes:
   - If a5 had ≥10% suspect-honest positives and you flipped most of them → `gap_f1` on Rot B drops by 5–15pp.
   - If only 2–3% were truly mislabelled → gap barely moves; the gap is then a prior/distribution issue, not noise. Move on to per-batch standardisation or group-stratified CV instead of more relabelling.
5. Track the residual gap (`cv_f1 − test_f1_at_cv`) before and after. That delta is your concrete number for whether the relabel work was worth it.

### What to do if BOTH a4 and a5 have similar suspect rates
That points away from "a5 is dirty" and toward systematic issues:
- The annotator standard drifted between batches.
- The class definition itself is fuzzy (the `genuine ambiguity` cases are over-represented).
- Action: tighten the labelling rubric *first*, then re-annotate a small audited subset with the new rubric, then expand.